# Sinh real/fake trên Kaggle

Mỗi lượt xử lý **một part clip sạch**: chỉnh mục 2, rồi chạy lần lượt các cell.

| | |
|---|---|
| **Đầu vào** | Part clip sạch đã review/export (CSV nguồn không dùng ở bước này) |
| **Đầu ra** | `vn-av-df-forensics/datasets/vn-av-df-data/<part>/` (real/full/partial/sham đang *pending*): *Save Version* rồi tải từ tab Output |
| **Thời gian** | Hết `TIME_BUDGET_HOURS` thì dừng nhận cặp mới, vẫn ghi W&B và dọn Output; phiên sau đặt `PREVIOUS_GENERATION` để sinh tiếp |
| **Kaggle** | Bật Internet + GPU. Worker MuseTalk chỉ cài khi danh sách generator có `musetalk_1_5` |

> Cần pilot thật để xác nhận checkpoint/worker trên Kaggle; kiểm thử hiện dùng media/generator fixture.

## 1. Cài đặt
Clone repository, cài `data_pipeline` và gói `[generation]`.

In [ ]:
from pathlib import Path
import subprocess, sys, os, time
SESSION_START = time.time()  # Mốc tính giới hạn ~12 giờ của phiên Kaggle.
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
# Môi trường worker (~10+ GB) để ngoài /kaggle/working: không tính vào Output 20 GB, mất khi hết phiên.
WORKERS = Path("/kaggle/temp/vn-av-df-workers")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
assert (ROOT / "settings.py").is_file(), "Cần upload/push code mới trước khi dùng notebook"
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT / "data_pipeline"), "-e", str(ROOT) + "[generation]"], check=True)
import settings
settings.bootstrap()
from vn_av_df.actions import execute

## 2. Chọn part và thông số
- Mỗi part chia độc lập 70/15/15 theo nhóm người/nguồn; không cần split-lock của part trước.
- `CLIPS_PER_SPLIT = 2` để chạy thử; `0` để lấy mọi parent đủ điều kiện. Số output lớn hơn số parent vì có real/full/partial và có thể sham.
- Plan đã tạo sẽ khóa cấu hình: đổi cấu hình cần plan/output mới, không sửa part đã chốt tại chỗ.
- `TIME_BUDGET_HOURS`: sau số giờ này (tính từ cell 1) không nhận cặp mới, để kịp ghi W&B, dọn Output và lưu trước giới hạn 12 giờ.
- `PREVIOUS_GENERATION`: phiên trước hết giờ chưa xong thì attach Output của nó (*Add Input → Your Work → Notebook Output*) và trỏ tới thư mục `vn-av-df-forensics` trong đó; cặp đã sinh được giữ nguyên, cặp dở được sinh lại.
- `GPUS = "auto"`: dùng mọi GPU (T4 x2 → 2 cặp song song, mỗi GPU một worker).

In [ ]:
PART = 1  # Ví dụ: 2 -> xử lý vn-av-df-data-part2.
GENERATION_NAME = f"vn-av-df-data-part{PART}"
CLEAN_DATASET = Path("/kaggle/input/YOUR_CLEAN_PART") / GENERATION_NAME
SPLIT_RATIOS = {"train": 0.7, "validation": 0.15, "test": 0.15}  # Tỷ lệ clip sạch; giữ nguyên nhóm.
# Tạm 70/15/15: part 1 (10 speaker) chia 80/10/10 thì validation chỉ có 1 người.
SPLIT_HISTORY = []  # Tạm tắt lịch sử giữa các part, kể cả PART >= 2.
# Bật lại: liệt kê các split-lock bao phủ mọi part trước; file tích lũy đủ thì chỉ cần một.
# SPLIT_HISTORY = [Path("/kaggle/input/PART1_LOCK/split-lock.json")]  # Ví dụ cho part 2.
DEVICE = "cuda"
SEED = 42  # Ngẫu nhiên của split/plan; giữ cố định giữa các part.
CLIPS_PER_SPLIT = 2  # 2 = pilot; 0 = toàn bộ clip đủ điều kiện trong từng split.
PARTIAL_SECONDS = [0.4, 0.8, 1.6, 2.4]  # Chọn ngẫu nhiên 1 độ dài fake cục bộ/parent.
CRF = 18                # Nén H.264 chung cho real/fake; số nhỏ hơn = file lớn hơn.
MAX_SIDE = 640          # Giới hạn cạnh dài của video output.
# Thiết kế 2×2. donor: miệng vẽ theo tiếng thật khác của cùng người (mối đe dọa chính).
# source: miệng vẽ theo đúng tiếng gốc (chỉ có artifact), đối chứng ở validation/test.
# Parent không tìm được donor cùng speaker_id bị bỏ qua và ghi vào plan.
FAKE_AUDIO_MODES_BY_SPLIT = {
    "train": ["donor"],
    "validation": ["donor", "source"],
    "test": ["donor", "source"],
}
INCLUDE_SHAM = True     # Sham: ghép tiếng thật cùng người, không AI (nhãn AI=0, có đoạn lệch A/V).
GENERATORS_BY_SPLIT = {
    "train": ["wav2lip_gan"],
    "validation": ["wav2lip_gan"],
    "test": ["wav2lip_gan", "musetalk_1_5"],
}
# Pilot chỉ Wav2Lip, không cài MuseTalk:
# GENERATORS_BY_SPLIT = {s: ["wav2lip_gan"] for s in ("train", "validation", "test")}
MUSETALK_BATCH_SIZE = 4  # Giảm nếu worker MuseTalk thiếu VRAM.
MUSETALK_SEED = 42
TIME_BUDGET_HOURS = 11  # Kaggle dừng phiên ở 12 giờ; chừa ~1 giờ cho W&B, dọn dẹp và lưu Output.
PREVIOUS_GENERATION = None  # Sinh tiếp, ví dụ Path("/kaggle/input/YOUR_PREVIOUS_OUTPUT/vn-av-df-forensics").
GPUS = "auto"  # Hoặc [0] để chỉ dùng một GPU.

## 3. Kiểm tra dữ liệu, lập plan, cài generator
Plan đã có thì chỉ kiểm tra khớp cấu hình; chưa có thì tạo mới.

In [ ]:
settings.PART, settings.GENERATION_NAME = PART, GENERATION_NAME
settings.CLEAN_DATASET, settings.SPLIT_HISTORY = CLEAN_DATASET, SPLIT_HISTORY
settings.SPLIT_RATIOS = SPLIT_RATIOS
settings.DEVICE = DEVICE
cfg = settings.config()
cfg["seed"] = SEED
cfg["generation"].update(
    clips_per_split=CLIPS_PER_SPLIT, partial_seconds=PARTIAL_SECONDS,
    crf=CRF, max_side=MAX_SIDE, fake_audio_modes_by_split=FAKE_AUDIO_MODES_BY_SPLIT,
    include_sham=INCLUDE_SHAM, generators_by_split=GENERATORS_BY_SPLIT,
)
cfg["musetalk"].update(batch_size=MUSETALK_BATCH_SIZE, seed=MUSETALK_SEED)
if PREVIOUS_GENERATION is not None:
    # Chép plan + output phiên trước vào đúng chỗ; generate bỏ qua cặp đã có record.
    import shutil
    plan_path = Path(cfg["plan"])
    for target in (plan_path, plan_path.with_name(plan_path.stem + "_split-lock.json"),
                   Path(cfg["generated_dataset"])):
        source = Path(PREVIOUS_GENERATION) / target.relative_to(settings.ROOT)
        assert source.exists(), f"Không thấy {source}; kiểm tra PREVIOUS_GENERATION"
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            (shutil.copytree if source.is_dir() else shutil.copy2)(source, target)
    print("Sinh tiếp từ:", PREVIOUS_GENERATION)
print("Part:", PART, "| Input:", cfg["clean_dataset"], "| Output:", cfg["generated_dataset"])
print("Thông số:", cfg["generation"])
print("Tỷ lệ split:", cfg["split_ratios"], "| Lịch sử:", cfg["split_history"])
execute("validate", cfg)
from vn_av_df.common.runtime import read_json
from vn_av_df.generation import validate_plan_settings
if Path(cfg["plan"]).exists():
    plan = read_json(cfg["plan"])
    validate_plan_settings(cfg, plan)
else:
    execute("plan", cfg)
execute("generator_setup", cfg)

In [ ]:
needs_musetalk = any("musetalk_1_5" in names for names in cfg["generation"]["generators_by_split"].values())
if needs_musetalk:
    try:
        subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "musetalk", "--root", str(ROOT), "--env-root", str(WORKERS)], check=True)
    except subprocess.CalledProcessError:
        # Lỗi thật nằm trong log cài đặt (pip/conda/import), không nằm trong CalledProcessError.
        log = WORKERS / ".kaggle-tools/musetalk/setup.log"
        if log.exists():
            print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
        raise RuntimeError(f"Cài worker MuseTalk lỗi; 80 dòng cuối của {log} ở trên") from None
    worker_python = str(WORKERS / ".venv-musetalk/bin/python")
    settings.MUSETALK_PYTHON = worker_python
    cfg["musetalk"]["python"] = worker_python
    print("Worker ready:", worker_python)
else:
    print("Không chọn MuseTalk: bỏ qua cài worker MuseTalk.")

## 4. W&B (tuỳ chọn)
Ghi cấu hình, số mẫu theo điều kiện và video mẫu lên Weights & Biases. Run mở ngay ở cell này nên tab *Logs*/*System* có log và GPU/CPU/RAM suốt lúc sinh.
- Lần đầu: thêm Kaggle secret `WANDB_API_KEY` (*Add-ons → Secrets*, bật cho notebook).
- Tắt: đặt `USE_WANDB = False`. Không có secret thì tự tắt, pipeline vẫn chạy.

In [ ]:
import json
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-forensics"
WANDB_ENTITY = None  # Team W&B; None = tài khoản của API key.
LOG_SAMPLE_VIDEOS = 6  # Số video mẫu đưa lên W&B; 0 = không.
if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        # Key qua biến môi trường (không truyền vào code); W&B tự đọc.
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login()
    except Exception as exc:  # Không có secret: tắt W&B, pipeline vẫn chạy.
        USE_WANDB = False
        print("Tắt W&B, không đọc được secret WANDB_API_KEY:", exc)

def plain(value):
    """Cấu hình/metric về kiểu JSON đơn giản để W&B nhận."""
    return json.loads(json.dumps(value, default=str))

def stamp():
    """Ngày giờ Việt Nam (máy Kaggle chạy UTC) gắn vào tên run để phân biệt các lần chạy."""
    from datetime import datetime, timedelta, timezone
    return datetime.now(timezone(timedelta(hours=7))).strftime("%m%d-%H%M")

def open_run():
    """Mở run trước khi sinh để W&B ghi log console và GPU/CPU/RAM suốt lúc sinh."""
    if wandb.run is not None:  # Chạy lại cell: đóng run cũ.
        wandb.finish()
    return wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="generate", group=GENERATION_NAME,
        name=f"generate-{GENERATION_NAME}-{stamp()}",
        config=plain({"part": PART, "seed": SEED, "split_ratios": SPLIT_RATIOS,
                      "generation": cfg["generation"],
                      "musetalk": {"batch_size": MUSETALK_BATCH_SIZE, "seed": MUSETALK_SEED}}),
    )

if USE_WANDB:
    run = open_run()

## 5. Sinh video
- Mỗi generator một worker sống suốt lượt sinh (nạp model một lần); mỗi GPU sinh một cặp song song.
- Log mỗi cặp: split, generator, parent/donor, thời gian, thời gian còn lại ước tính. Không tự đổi generator khi có lỗi.
- Hết `TIME_BUDGET_HOURS`: dừng nhận cặp mới, báo `status: partial`; các cell sau vẫn chạy (W&B, dọn dẹp) để *Save Version* giữ phần đã sinh.
- Lỗi khi sinh: vẫn chạy W&B và dọn dẹp, lỗi được báo lại ở cell cuối.
- Output còn *pending*, chưa dùng để train cho đến khi review/finalize.

In [ ]:
import torch, traceback
cfg["generation_gpus"] = list(range(torch.cuda.device_count())) if GPUS == "auto" else list(GPUS)
cfg["generation_deadline"] = SESSION_START + TIME_BUDGET_HOURS * 3600
print("GPU:", cfg["generation_gpus"] or "không có (tuần tự)",
      "| dừng nhận cặp mới sau:", time.strftime("%H:%M", time.localtime(cfg["generation_deadline"])), "(giờ máy Kaggle)")
generate_error = None
try:
    execute("generate", cfg)
except Exception as exc:  # Vẫn ghi W&B + dọn Output để giữ phần đã sinh; báo lỗi ở cell cuối.
    generate_error = exc
    traceback.print_exc()

## 6. Ghi kết quả lên W&B
Bỏ qua nếu `USE_WANDB = False`. Ghi vào run đang mở từ mục 4:
- `split/*`: số clip, speaker, nguồn, phút và tỷ lệ thực tế mỗi split; `speakers_in_multiple_splits` phải bằng 0.
- `plan/*`: từng cặp (parent, donor, generator, audio_mode, sham donor) và parent bị bỏ vì không có donor.
- Số mẫu theo split × variant × generator × audio_mode; histogram độ dài/vị trí đoạn fake cục bộ và đoạn ghép sham.
- Phiên bản/checkpoint generator, bảng mẫu và vài video.

In [ ]:
# Bổ sung kết quả sinh vào run đang mở rồi đóng run: chia split, cặp sinh, số mẫu theo điều kiện,
# đoạn fake cục bộ, generator, bảng mẫu và vài video.
out = Path(cfg["generated_dataset"])
if USE_WANDB and not (out / "candidates.jsonl").exists():
    # Chưa đủ cặp (hết giờ hoặc lỗi): chỉ ghi tiến độ, bảng đầy đủ ghi ở phiên hoàn tất.
    plan = json.loads(Path(cfg["plan"]).read_text(encoding="utf-8"))
    run = wandb.run or open_run()
    run.summary.update({"status": "partial", "pairs_total": len(plan["jobs"]),
                        "pairs_done": len(list((out / "records").glob("*.json")))})
    run.finish()
elif USE_WANDB:
    import pandas as pd
    text = (out / "candidates.jsonl").read_text(encoding="utf-8")
    df = pd.DataFrame([json.loads(x) for x in text.splitlines() if x.strip()])
    plan = json.loads(Path(cfg["plan"]).read_text(encoding="utf-8"))
    run = wandb.run or open_run()  # Run mở từ mục 4; đã đóng (chạy lại cell) thì mở run mới.
    # Phiên bản/checkpoint generator thật sự đã dùng (generation.json).
    generation = out / "generation.json"
    if generation.exists():
        run.config.update(plain({"provenance": json.loads(generation.read_text(encoding="utf-8"))["provenance"]}),
                          allow_val_change=True)

    # Chia split của clip sạch: kiểm tra speaker/nguồn không lẫn giữa split và tỷ lệ thực tế.
    clean = pd.DataFrame(plan["assignments"])
    per_split = clean.groupby("split").agg(clips=("clip_id", "count"), speakers=("speaker_id", "nunique"),
                                           sources=("source_id", "nunique"), seconds=("duration_s", "sum")).reset_index()
    per_split["minutes"] = (per_split.pop("seconds") / 60).round(2)
    per_split["ratio"] = (per_split["clips"] / per_split["clips"].sum()).round(3)
    per_speaker = clean.groupby(["speaker_id", "split"]).agg(clips=("clip_id", "count")).reset_index()
    leaked = clean.groupby("speaker_id")["split"].nunique()
    jobs = pd.DataFrame([{
        "split": j["original"]["split"], "parent": j["original"]["clip_id"],
        "speaker_id": j["original"].get("speaker_id"), "generator": j["generator"],
        "audio_mode": j["audio_mode"], "donor": j["donor"]["clip_id"],
        "sham_donor": (j.get("sham_donor") or {}).get("clip_id"), "emit_real": j.get("emit_real", True),
    } for j in plan["jobs"]])

    df["audio_mode"] = df["audio_mode"].fillna("-")
    keys = ["split", "variant", "generator", "audio_mode"]
    counts = df.groupby(keys).size().reset_index(name="samples")
    columns = ["sample_id", *keys, "label", "duration_s", "speaker_id", "source_id",
               "audio_source_id", "fake_intervals", "av_mismatch_intervals"]
    # Đoạn fake cục bộ (partial) và đoạn ghép sham: độ dài và vị trí trong clip.
    # Làm tròn: giá trị lệch nhau cỡ 1e-9 làm wandb.Histogram báo lỗi.
    spans = [{"variant": r.variant, "length_s": round(b - a, 3), "start_s": round(a, 3),
              "start_ratio": round(a / r.duration_s, 3)}
             for r in df.itertuples() if r.variant in ("partial", "sham")
             for a, b in (r.fake_intervals if r.variant == "partial" else r.av_mismatch_intervals) or []]
    spans = pd.DataFrame(spans)
    run.log({
        "split/per_split": wandb.Table(dataframe=per_split),
        "split/per_speaker": wandb.Table(dataframe=per_speaker),
        "plan/pairs": wandb.Table(dataframe=jobs.astype(str)),
        "plan/skipped_without_donor": wandb.Table(columns=["clip_id"], data=[[c] for c in plan["skipped_without_donor"]]),
        "samples_by_condition": wandb.Table(dataframe=counts),
        "samples": wandb.Table(dataframe=df[columns].astype(str)),
    })
    if len(spans):
        run.log({"spans/table": wandb.Table(dataframe=spans),
                 **{f"spans/{v}_length_s": wandb.Histogram(g["length_s"]) for v, g in spans.groupby("variant")},
                 **{f"spans/{v}_start_ratio": wandb.Histogram(g["start_ratio"]) for v, g in spans.groupby("variant")}})
    run.summary.update({
        "status": "complete", "samples": len(df), "fake": int(df["label"].sum()), "real": int((df["label"] == 0).sum()),
        **{f"variant_{k}": int(v) for k, v in df["variant"].value_counts().items()},
        "pairs": len(plan["jobs"]), "skipped_without_donor": len(plan["skipped_without_donor"]),
        **{f"pairs_{k}": v for k, v in plan["split_counts"].items()},
        "clean_clips": len(clean), "speakers": int(clean["speaker_id"].nunique()),
        # Phải bằng 0: một speaker chỉ thuộc một split.
        "speakers_in_multiple_splits": int((leaked > 1).sum()),
        **{f"clean_ratio_{r.split}": float(r.ratio) for r in per_split.itertuples()},
    })
    # Mỗi (variant, generator, audio_mode) một video, tối đa LOG_SAMPLE_VIDEOS.
    sample = df.sort_values("sample_id").groupby(keys[1:]).head(1).head(LOG_SAMPLE_VIDEOS)
    if len(sample):
        run.log({"videos": [
            wandb.Video(str(out / r.video), format="mp4",
                        caption=f"{r.variant} | {r.generator} | {r.audio_mode} | {r.sample_id}")
            for r in sample.itertuples()
        ]})
    run.finish()

## 7. Dọn Output
Kaggle chỉ lưu tối đa 20 GB `/kaggle/working`. Xoá môi trường worker, source và model (cài lại được ở phiên sau), chỉ giữ dataset đã sinh, plan và log môi trường.

In [ ]:
import shutil
# Môi trường worker đã ở WORKERS (ngoài Output); chỉ còn source/weights tải lại được.
for name in ("external", "weights", "cache"):
    path = ROOT / name
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)
        print("Đã xoá", path)

def size_gb(path):
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1e9

print(f"Còn lại trong /kaggle/working: {size_gb(Path('/kaggle/working')):.2f} GB")
print("Dataset:", cfg["generated_dataset"], f"({size_gb(Path(cfg['generated_dataset'])):.2f} GB)")
print("Plan:", cfg["plan"])
if generate_error is not None:
    raise generate_error  # Lỗi sinh ở mục 5, báo lại sau khi đã giữ phần đã sinh.

## 8. Review/finalize ở local
1. *Save Version*. Nếu mục 5 báo `status: partial`: tạo phiên mới, attach Output này, đặt `PREVIOUS_GENERATION`, chạy lại cho đến khi `status: complete`.
2. Tải thư mục `vn-av-df-forensics/datasets/vn-av-df-data/vn-av-df-data-partN/` từ tab Output, đặt vào `datasets/vn-av-df-data/` của repo ở local.
3. Đặt `PART=N` trong `data_settings.py`, chạy `generation/04_review.py` → `05_finalize.py` → `07_export.py`.
4. Upload ZIP đã finalize sang Kaggle; chọn part đó (hoặc gộp nhiều part) trong `train.ipynb`.

`split-lock.json` (trong thư mục part) ghi cách chia part; mặc định không cần đưa sang part kế tiếp.